In [77]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [78]:
ruta_OF = os.path.join("res\\raw", "CPRManufacturingOrder.csv")
print(ruta_OF)

res\raw\CPRManufacturingOrder.csv


In [79]:
columns_OF = ["IDManufacturingOrder", "CodCompany",	"CodManufacturingOrder", "IDArticle", "IDStructure", "Description", "TimeUnitDefault",	"IDStructureType",	"Notes", "Quantity", "IDWareHouse", "PlannedEndDate", "PlannedStartDate", "PlannedEndMRPDate", "ManualEndDate", "Priority", "IDMOSituation", "Color", "Group", "InternalCode", "IDRoute", "AcceptedDate", "RealEndDate", "IDProject", "IDProTask", "IDProMaterial", "AutomaticCodification", "ManualPStartDate", "ManualPEndDate", "IDSite", "Historic", "RowTimestamp", "PlanningDirection", "LastModifiedBy", "QuantityBaseFormula", "IDUnitBaseFormula", "IDArticleVersion", "CreationTimestamp", "RowCreatedBy", "IDMaintenanceOrder", "IDOrderMaterialPrevision",	"IDLaunchedMRP", "AutoBatchAssignSalesOrders"]

pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None)        

df_OF = pd.read_csv(ruta_OF, sep=";", header=None, names=columns_OF)
print(df_OF.shape)

C:\Users\jaume\AppData\Local\Temp\ipykernel_20900\264357410.py:6: DtypeWarning: Columns (0: Group) have mixed types. Specify dtype option on import or set low_memory=False.
  df_OF = pd.read_csv(ruta_OF, sep=";", header=None, names=columns_OF)


(20659, 43)


In [80]:
df_OF = df_OF[df_OF["CodCompany"] == 100]
print(df_OF.shape)
display(df_OF.sample(1))

(18055, 43)


,IDManufacturingOrder,CodCompany,CodManufacturingOrder,IDArticle,IDStructure,Description,TimeUnitDefault,IDStructureType,Notes,Quantity,IDWareHouse,PlannedEndDate,PlannedStartDate,PlannedEndMRPDate,ManualEndDate,Priority,IDMOSituation,Color,Group,InternalCode,IDRoute,AcceptedDate,RealEndDate,IDProject,IDProTask,IDProMaterial,AutomaticCodification,ManualPStartDate,ManualPEndDate,IDSite,Historic,RowTimestamp,PlanningDirection,LastModifiedBy,QuantityBaseFormula,IDUnitBaseFormula,IDArticleVersion,CreationTimestamp,RowCreatedBy,IDMaintenanceOrder,IDOrderMaterialPrevision,IDLaunchedMRP,AutoBatchAssignSalesOrders
19938,f7045819-d9f9-4d71-b9fa-8759d122044f,100,OF2500336,682b014d-2ef4-4969-a224-e9c2e0eddf25,1ca7a0ee-b546-4056-8622-3a15d062fce3,Elektrodenaufnahme,2,d3bd7234-a0af-4c35-b658-dcd8ac943c67,<p>A2</p>,4.0,8dc40f3a-f875-4dbf-8844-63282db4196f,2025-01-31 21:00:00.000,2025-01-29 15:00:00.000,2025-02-11 00:00:00.000,2025-02-11 00:00:00.000,0,f57597bd-79b3-4a87-9f62-fdd434de7804,-1457710.0,NaN,CV250197/0198/PE1XA2X0026/1,cf345109-6d07-423f-9a2e-bf48b8c569c8,2025-01-17 00:00:00.000,2025-02-04 00:00:00.000,NaN,NaN,NaN,1,2025-02-10 19:30:00.000,2025-02-11 00:00:00.000,5565bc54-9d9c-4dc7-91e7-bb5e206c9bd7,0,2026-06-15 17:45:45.460,1,JMAU,4.0,153b2e67-a636-4e4f-9fa5-fc100da76dfb,NaN,2025-01-17 10:39:44.447,AFER,NaN,NaN,fa36b4bd-0b87-4d5f-86f1-34c24dc6be58,0


# Limpieza de OF.
Hay muchas ordenes de fabricación, para que el modelo tenga sentido debó moldearlo con los tiempos reales totales, es decir, no me sirve trabajar con OF que no estan terminadas, o que tienen imputaciones a medias. 

Esto lo haré de dos formas:

1. Descartando todas las OF las cuales su estado (IDMOSituation) sea "Creada". Ya que OF con este estado nunca han recivido imputaciones de tiempo.

2. Descartando todas las OF las cuales su fecha de finalización (RealEndDate) sea antes del 01-08-2026. 

## 1. Descartar OF por su estado
Las fases que su estado sea creadas debemos eliminarlas, el estado "Creada" puede tener dos ID:    
    - ID 100: 7776bcc3-98be-48f1-913a-30bbb40ef0bb   
    - ID 200: 7776bcc3-98be-48f1-913a-30bbb40ef0bb200   

In [81]:
df_OF_Creada = df_OF[df_OF['IDMOSituation'].isin(["7776bcc3-98be-48f1-913a-30bbb40ef0bb", "7776bcc3-98be-48f1-913a-30bbb40ef0bb200"])]
print(f"Total de OFs con estado Creadas (eliminar): {df_OF_Creada.shape[0]}")

df_OF = df_OF[~df_OF['IDMOSituation'].isin(["7776bcc3-98be-48f1-913a-30bbb40ef0bb", "7776bcc3-98be-48f1-913a-30bbb40ef0bb200"])]
print(f"Total de OFs tras filtrar por situación: {df_OF.shape[0]}")

Total de OFs con estado Creadas (eliminar): 1
Total de OFs tras filtrar por situación: 18054


## 2. Decartar OF por su fecha 
- **Decartar OF por su fecha finalización**      

Aquellas OF con fecha de finalización prevista posterior al 01/08/2026 las descartaremos ya que la totalidad de sus fases no estarán terminadas. 

Esto lo hare con la columna `ManualEndDate`. 

Con el equipo técnico hemos determinado que a partir de agosto, no se puede garantizar la finalización de todas sus fases, por ende vamos a prescindir de todas las OFs que tengan la fecha de finalización en agosto.      

In [82]:
df_OF["CreationTimestamp"] = pd.to_datetime(df_OF["CreationTimestamp"])
print(f"Total OFs {df_OF.shape[0]}")

df_OF_validas = df_OF[df_OF["CreationTimestamp"] >= "2025-06-01"]
print(f"Total OFs válidas {df_OF_validas.shape[0]}")
print(f"Total de OFs fuera de fecha {df_OF.shape[0] - df_OF_validas.shape[0]}")

Total OFs 18054
Total OFs válidas 12751
Total de OFs fuera de fecha 5303


- **Decartar OF por su fecha creación**
Aquellas OF con fecha de creación previa al 01/06/2025 las descartaremos ya que el sistema estaba en fase de implementación, esos datos no son fiables. 


In [83]:
df_OF["ManualEndDate"] = pd.to_datetime(df_OF["ManualEndDate"])
print(f"Total OFs {df_OF.shape[0]}")

df_OF_validas = df_OF[df_OF["ManualEndDate"] < "2026-08-01"]
print(f"Total OFs válidas {df_OF_validas.shape[0]}")
print(f"Total de OFs fuera de fecha {df_OF.shape[0] - df_OF_validas.shape[0]}")

Total OFs 18054
Total OFs válidas 17284
Total de OFs fuera de fecha 770


## 3. Limpieza de Atributos
Vamos a prescindir de los atributos innecesarios. 

In [84]:
def eliminar_columnas(df):
    """Elimina de df_OF las columnas marcadas como innecesarias.
    Devuelve un DataFrame nuevo (no modifica el original)."""

    cols_sobran = [
        'IDStructure', 'TimeUnitDefault','Notes', 'IDWareHouse', 'Priority', 
        'Color', 'Group', 'InternalCode', 'IDRoute', 'AcceptedDate', 'CodCompany'
        'IDProject', 'IDProTask', 'IDProMaterial', 'AutomaticCodification',
        'Historic', 'RowTimestamp', 'PlanningDirection', 'LastModifiedBy',
        'IDMaintenanceOrder', 'IDOrderMaterialPrevision', 'IDLaunchedMRP', 
        'AutoBatchAssignSalesOrders', 'RowCreatedBy', 'CreationTimestamp', 
        'IDArticleVersion', 'IDSite', 'PlannedEndDate', 'PlannedStartDate',	
        'PlannedEndMRPDate', 'ManualPStartDate', 'ManualPEndDate', 'ManualEndDate'	 
    ]

    cols_a_quitar = [c for c in cols_sobran if c in columns_OF]
    df_out = df.drop(columns=cols_a_quitar)

    # Aviso por si alguna de la lista no se encontró (typo o nombre distinto)
    no_encontradas = [c for c in cols_sobran if c not in df_OF.columns]
    if no_encontradas:
        print(f"Aviso: {len(no_encontradas)} columnas de la lista no estaban en df: {no_encontradas}")

    print(f"Eliminadas {len(cols_a_quitar)} columnas. "
          f"Quedan {df_out.shape[1]} columnas, {df_out.shape[0]} filas.")
    return df_out

In [85]:
df_OF = eliminar_columnas(df_OF_validas)

Aviso: 1 columnas de la lista no estaban en df: ['CodCompanyIDProject']
Eliminadas 31 columnas. Quedan 12 columnas, 17284 filas.


In [86]:
display(df_OF.head(1))

,IDManufacturingOrder,CodCompany,CodManufacturingOrder,IDArticle,Description,IDStructureType,Quantity,IDMOSituation,RealEndDate,IDProject,QuantityBaseFormula,IDUnitBaseFormula
0,000744d2-5b9e-40be-9126-9a8a66954f33,100,OF2502249,921ac292-f882-47f8-ae0b-ce82c4f1a3c8,CONDUCTEUR 103127410,d3bd7234-a0af-4c35-b658-dcd8ac943c67,7.0,f57597bd-79b3-4a87-9f62-fdd434de7804,2025-04-17 00:00:00.000,NaN,7.0,153b2e67-a636-4e4f-9fa5-fc100da76dfb


Guardamos el `df_OF_validas` en el repositorio res//procesed

In [87]:
df_OF.to_parquet("res//procesed/df_OF.parquet")